# NLP Assignment P04
## Feature Extraction / Ekstraksi Fitur

Notebook ini digunakan di Google Colab untuk mengubah corpus teks menjadi fitur numerik menggunakan Bag of Words dan TF-IDF, kemudian mencari dokumen yang paling mirip dengan cosine similarity.

## Tujuan

1. Melakukan preprocessing pada corpus.
2. Membuat matriks Bag of Words.
3. Membuat matriks TF-IDF.
4. Menghitung cosine similarity untuk sebuah query.
5. Menganalisis perbedaan fitur sparse dan embedding.

## 1. Instalasi dan import library

Google Colab biasanya sudah menyediakan NumPy, pandas, dan scikit-learn.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib seaborn

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('NumPy:', np.__version__)
print('pandas:', pd.__version__)

## 2. Menyiapkan corpus

Corpus terdiri dari 10 dokumen pendek tentang Python, data science, machine learning, dan AI.

In [ ]:
sentences = [
    'Saya suka belajar data science.',
    'Python adalah bahasa pemrograman yang populer.',
    'Saya menggunakan Python untuk analisis data.',
    'Analisis data membantu dalam pengambilan keputusan.',
    'Machine learning adalah cabang dari kecerdasan buatan.',
    'Algoritma machine learning dapat memprediksi hasil.',
    'Saya tertarik pada teknologi baru.',
    'Kecerdasan buatan memiliki banyak aplikasi.',
    'Belajar data science sangat menarik.',
    'Saya mengikuti kursus online tentang machine learning.',
]

corpus = pd.DataFrame({'document_id': range(1, len(sentences) + 1), 'text': sentences})
corpus

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

corpus['processed_text'] = corpus['text'].map(preprocess_text)
corpus[['document_id', 'text', 'processed_text']]

## 3. Bag of Words

Bag of Words menghitung frekuensi setiap token pada setiap dokumen. Representasinya bersifat sparse dan tidak menyimpan urutan kata.

In [ ]:
bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(corpus['processed_text'])
bow_features = bow_vectorizer.get_feature_names_out()
bow_table = pd.DataFrame(X_bow.toarray(), columns=bow_features)

print('Ukuran matriks BoW:', X_bow.shape)
print('Jumlah vocabulary:', len(bow_features))
display(bow_table)

## 4. TF-IDF

TF-IDF memberi bobot lebih tinggi pada kata yang penting dalam dokumen tertentu dan lebih rendah pada kata yang muncul di banyak dokumen.

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(corpus['processed_text'])
tfidf_features = tfidf_vectorizer.get_feature_names_out()
tfidf_table = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_features)

print('Ukuran matriks TF-IDF:', X_tfidf.shape)
print('Jumlah vocabulary:', len(tfidf_features))
display(tfidf_table.round(3))

In [ ]:
# Visualisasi 15 fitur TF-IDF dengan nilai rata-rata tertinggi
mean_tfidf = tfidf_table.mean().sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 5))
sns.barplot(x=mean_tfidf.values, y=mean_tfidf.index)
plt.title('Rata-rata Bobot TF-IDF Tertinggi')
plt.xlabel('Rata-rata TF-IDF')
plt.ylabel('Token')
plt.show()

## 5. Cosine similarity

Query harus menggunakan vectorizer yang sama dengan corpus. Vectorizer hanya di-`fit` pada corpus, sedangkan query diproses dengan `transform`.

In [ ]:
query = 'data'
query_vector = tfidf_vectorizer.transform([preprocess_text(query)])
scores = cosine_similarity(query_vector, X_tfidf).ravel()
ranking = np.argsort(scores)[::-1]

results = corpus[['document_id', 'text']].copy()
results['similarity'] = scores
results = results.sort_values('similarity', ascending=False).reset_index(drop=True)
print(f'Top dokumen untuk query: {query!r}')
display(results.head(5))

In [ ]:
assert X_bow.shape[0] == len(sentences)
assert X_tfidf.shape[0] == len(sentences)
assert len(scores) == len(sentences)
assert np.all(scores >= -1e-9) and np.all(scores <= 1 + 1e-9)
print('Validasi matriks dan skor similarity: PASS')

## 6. Analisis hasil

BoW menghasilkan nilai frekuensi token, sedangkan TF-IDF menghasilkan bobot kepentingan token. Dokumen dengan skor cosine similarity tertinggi memiliki fitur yang paling dekat dengan query `data`.

BoW dan TF-IDF cocok sebagai baseline karena cepat dan mudah diinterpretasikan. Keterbatasannya adalah tidak memahami sinonim, urutan kata, dan konteks. Word embedding menghasilkan vektor dense yang lebih semantik, sedangkan Transformer embedding lebih kontekstual tetapi membutuhkan resource lebih besar.

Vectorizer harus di-`fit` hanya pada data training atau corpus basis pencarian. Data testing dan query cukup diproses dengan `transform` agar tidak terjadi data leakage.

## 7. Kesimpulan

Feature extraction mengubah teks menjadi fitur numerik untuk machine learning dan pencarian dokumen. BoW merupakan baseline sederhana berbasis frekuensi, sedangkan TF-IDF mempertimbangkan kelangkaan token di seluruh corpus.

Cosine similarity dapat memberi peringkat dokumen berdasarkan kemiripan dengan query. Untuk query `data`, dokumen tentang data science dan analisis data diharapkan memperoleh skor tertinggi. Pengembangan berikutnya dapat membandingkan hasil TF-IDF dengan Sentence Transformers untuk semantic search.

In [ ]:
# Membuat draft analisis dan kesimpulan berdasarkan hasil eksperimen
top_document = results.iloc[0]
nonzero_bow = int((X_bow.toarray() > 0).sum())
nonzero_tfidf = int((X_tfidf.toarray() > 0).sum())

analysis = f'''
ANALISIS OTOMATIS
Eksperimen menggunakan {len(sentences)} dokumen dan menghasilkan {len(bow_features)} fitur vocabulary.
Matriks Bag of Words berukuran {X_bow.shape} dengan {nonzero_bow} nilai non-zero.
Matriks TF-IDF berukuran {X_tfidf.shape} dengan {nonzero_tfidf} nilai non-zero.
BoW menunjukkan frekuensi kemunculan token, sedangkan TF-IDF memberi bobot
lebih besar pada token yang relatif penting dan tidak terlalu umum dalam corpus.
Untuk query '{query}', dokumen dengan similarity tertinggi adalah:
'{top_document['text']}' dengan skor {top_document['similarity']:.4f}.
Hal ini menunjukkan bahwa cosine similarity menemukan dokumen yang memiliki
fitur kata paling dekat dengan query.
'''

conclusion = f'''
KESIMPULAN OTOMATIS
Feature extraction berhasil mengubah {len(sentences)} dokumen teks menjadi
representasi numerik menggunakan BoW dan TF-IDF. BoW cocok sebagai baseline
karena sederhana dan mudah diinterpretasikan, sedangkan TF-IDF lebih baik
dalam membedakan kata yang penting dari kata yang terlalu umum.

Berdasarkan cosine similarity, dokumen teratas untuk query '{query}' adalah
'{top_document['text']}' dengan skor {top_document['similarity']:.4f}.
Untuk pencarian berbasis makna yang lebih kompleks, eksperimen berikutnya
dapat membandingkan TF-IDF dengan word embedding atau Transformer embedding.
'''

print(analysis)
print(conclusion)